In [9]:
import pandas as pd
import re
#file_path = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240751-754.sum'
def process_sum_file(file_path):
    # Read the .sum file content
    with open(file_path, 'r') as file:
        content = file.read()
    
    # Extract the relevant sections using regular expressions
    title_match = re.search(r'^(240748-751 830 K)$', content, re.MULTILINE)
    atom_params_match = re.search(r'(Name\s+x\s+sx\s+y\s+sy\s+z\s+sz\s+B\s+sB\s+occ\.\ssocc\.\s+Mult\n.*?)(?=\n\S|$)', content, re.DOTALL)
    cell_params_match = re.search(r'(?<=\n)(\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n)', content)
    overall_tem_factor_match = re.search(r'=> Overall tem\. factor\s+:\s+(\d+\.\d+)\s+(\d+\.\d+)', content)
    r_factors_match = re.search(r'=> Rp:\s+(\d+\.\d+)\s+Rwp:\s+(\d+\.\d+)\s+Rexp:\s+(\d+\.\d+)\s+Chi2:\s+(\d+\.\d+)', content)
    bragg_r_factor_match = re.search(r'=> Bragg R-factor:\s+(\d+\.\d+)\s+Vol:\s+(\d+\.\d+\s*\(\s*\d+\.\d+\))\s+Fract\(%\):\s+(\d+\.\d+\s*\(\s*\d+\.\d+\))', content)

    # Create a dictionary to hold the data
    data = {
        'Title': [title_match.group(0) if title_match else None],
        'ATOM PARAMETERS': [atom_params_match.group(1) if atom_params_match else None],
        'Cell parameters': [cell_params_match.group(1).strip() if cell_params_match else None],
        'Overall tem. factor': [f"{overall_tem_factor_match.group(1)} {overall_tem_factor_match.group(2)}" if overall_tem_factor_match else None],
        'R factors': [f"{r_factors_match.group(1)} {r_factors_match.group(2)} {r_factors_match.group(4)}" if r_factors_match else None],
        'Bragg R-factor': [f"{bragg_r_factor_match.group(1)} {bragg_r_factor_match.group(2)} {bragg_r_factor_match.group(3)}" if bragg_r_factor_match else None]
    }

    # Create the DataFrame
    df = pd.DataFrame(data)

    # Organize the 'Title' column
    def organize_title(title):
        if title:
            parts = title.split('-')
            if len(parts) == 2:
                running_number = parts[0].strip()
                temperature = parts[1].strip().replace('K', '').replace(' ', '-')
                if temperature.endswith('-'):
                    temperature = temperature[:-1]
                return running_number, temperature
        return None, None

    # Apply the function and create new columns
    df[['Running Number of Data File', 'Temperature of measurement']] = df['Title'].apply(
        lambda x: pd.Series(organize_title(x))
    )

    # Drop the original 'Title' column
    df.drop(columns=['Title'], inplace=True)

    # Function to parse ATOM PARAMETERS and create a DataFrame
    def parse_atom_parameters(atom_params):
        lines = atom_params.split('\n')
        columns = ['Name', 'x', 'sx', 'y', 'sy', 'z', 'sz', 'B', 'sB', 'occ', 'socc', 'Mult']
        data = []
        for line in lines[1:]:  # Skip the header line
            parts = re.split(r'\s+', line.strip())
            if len(parts) == 12:
                data.append(parts)
        
        atom_df = pd.DataFrame(data, columns=columns)
        
        organized_data = {}
        
        for _, row in atom_df.iterrows():
            name = row['Name']
            organized_data[f'x({name})'] = row['x']
            organized_data[f'ESD(x({name}))'] = row['sx']
            organized_data[f'y({name})'] = row['y']
            organized_data[f'ESD(y({name}))'] = row['sy']
            organized_data[f'z({name})'] = row['z']
            organized_data[f'ESD(z({name}))'] = row['sz']
            organized_data[f'B({name})'] = row['B']
            organized_data[f'ESD(B({name}))'] = row['sB']
            organized_data[f'Occupation factor({name})'] = row['occ']
            organized_data[f'ESD(occupation factor({name}))'] = row['socc']
        
        return pd.DataFrame([organized_data])

    # Apply the parsing function to the 'ATOM PARAMETERS' column and merge with the main DataFrame
    atom_parameters_df = df['ATOM PARAMETERS'].apply(parse_atom_parameters)
    atom_parameters_df = pd.concat(atom_parameters_df.values).reset_index(drop=True)

    # Drop the original 'ATOM PARAMETERS' column and join the new organized DataFrame
    df.drop(columns=['ATOM PARAMETERS'], inplace=True)
    df = pd.concat([df, atom_parameters_df], axis=1)

    # Function to parse and organize 'Cell parameters'
    def organize_cell_parameters(cell_params):
        lines = cell_params.strip().split('\n')
        organized_data = {}
        
        if len(lines) == 6:
            organized_data['Unit cell parameter a'] = lines[0].strip()
            organized_data['Unit cell parameter b'] = lines[1].strip()
            organized_data['Unit cell parameter c'] = lines[2].strip()
            organized_data['Unit cell parameter alpha'] = lines[3].strip()
            organized_data['Unit cell parameter beta'] = lines[4].strip()
            organized_data['Unit cell parameter gamma'] = lines[5].strip()
        
        return pd.DataFrame([organized_data])

    # Apply the function to the 'Cell parameters' column and merge with the main DataFrame
    cell_parameters_df = df['Cell parameters'].apply(organize_cell_parameters)
    cell_parameters_df = pd.concat(cell_parameters_df.values).reset_index(drop=True)

    # Drop the original 'Cell parameters' column and join the new organized DataFrame
    df.drop(columns=['Cell parameters'], inplace=True)
    df = pd.concat([df, cell_parameters_df], axis=1)

    # Function to parse and organize 'Overall tem. factor'
    def organize_overall_tem_factor(overall_tem_factor):
        parts = overall_tem_factor.split()
        if len(parts) == 2:
            return pd.Series({'Bover': parts[0], 'ESD(Bover)': parts[1]})
        return pd.Series({'Bover': None, 'ESD(Bover)': None})

    # Apply the function to the 'Overall tem. factor' column and merge with the main DataFrame
    overall_tem_factor_df = df['Overall tem. factor'].apply(organize_overall_tem_factor)
    df = pd.concat([df, overall_tem_factor_df], axis=1)
    df.drop(columns=['Overall tem. factor'], inplace=True)

    # Function to parse and organize 'R factors'
    def organize_r_factors(r_factors):
        parts = r_factors.split()
        if len(parts) == 4:
            return pd.Series({'Rp': parts[0], 'Rwp': parts[1], 'Chi^2': parts[3]})
        return pd.Series({'Rp': None, 'Rwp': None, 'Chi^2': None})

    # Apply the function to the 'R factors' column and merge with the main DataFrame
    r_factors_df = df['R factors'].apply(organize_r_factors)
    df = pd.concat([df, r_factors_df], axis=1)
    df.drop(columns=['R factors'], inplace=True)

    # Function to parse and organize 'Bragg R-factor'
    def organize_bragg_r_factor(bragg_r_factor):
        parts = re.split(r'\s+', bragg_r_factor.strip())
        if len(parts) >= 7:
            volume = re.findall(r'\d+\.\d+', parts[3])
            fraction = re.findall(r'\d+\.\d+', parts[5])
            return pd.Series({
                'RBragg': parts[0],
                'Unit cell volume V': volume[0],
                'ESD(V)': volume[1],
                'Phase fraction w': fraction[0],
                'ESD(w)': fraction[1]
            })
        return pd.Series({
            'RBragg': None,
            'Unit cell volume V': None,
            'ESD(V)': None,
            'Phase fraction w': None,
            'ESD(w)': None
        })

    # Apply the function to the 'Bragg R-factor' column and merge with the main DataFrame
    bragg_r_factor_df = df['Bragg R-factor'].apply(organize_bragg_r_factor)
    df = pd.concat([df, bragg_r_factor_df], axis=1)
    df.drop(columns=['Bragg R-factor'], inplace=True)

    return df

# List of .sum files to process
sum_files = [
    r'C:\\Users\\Master.WR\\OneDrive - stud.tu-darmstadt.de\\Document\\240751-754.sum',
    r'C:\\Users\Master.WR\\OneDrive - stud.tu-darmstadt.de\\Documents\\240754-757.sum',
    r'C:\\Users\Master.WR\\OneDrive - stud.tu-darmstadt.de\\Documents\\240757-760.sum',
    r'C:\\Users\Master.WR\\OneDrive - stud.tu-darmstadt.de\\Documents\\240760-763.sum',
    r'C:\\Users\Master.WR\\OneDrive - stud.tu-darmstadt.de\\Documents\\240763-766.sum'
]

for file_path in sum_files:
    df = process_sum_file(file_path)
    csv_path = file_path.replace('.sum', '.csv')
    df.to_csv(csv_path, index=False)
    print(f"Processed and saved: {csv_path}")



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\\\Users\\\\Master.WR\\\\OneDrive - stud.tu-darmstadt.de\\\\Document\\\\240751-754.sum'

In [3]:
import pandas as pd
import re
import os

def process_sum_file(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None
    
    # Read the .sum file content
    try:
        with open(file_path, 'r') as file:
            content = file.read()
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return None
    
    # Extract the relevant sections using regular expressions
    title_match = re.search(r'^(240748-751 830 K)$', content, re.MULTILINE)
    atom_params_match = re.search(r'(Name\s+x\s+sx\s+y\s+sy\s+z\s+sz\s+B\s+sB\s+occ\.\ssocc\.\s+Mult\n.*?)(?=\n\S|$)', content, re.DOTALL)
    cell_params_match = re.search(r'(?<=\n)(\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n\s*\d+\.\d+\s+\d+\.\d+\n)', content)
    overall_tem_factor_match = re.search(r'=> Overall tem\. factor\s+:\s+(\d+\.\d+)\s+(\d+\.\d+)', content)
    r_factors_match = re.search(r'=> Rp:\s+(\d+\.\d+)\s+Rwp:\s+(\d+\.\d+)\s+Rexp:\s+(\d+\.\d+)\s+Chi2:\s+(\d+\.\d+)', content)
    bragg_r_factor_match = re.search(r'=> Bragg R-factor:\s+(\d+\.\d+)\s+Vol:\s+(\d+\.\d+\s*\(\s*\d+\.\d+\))\s+Fract\(%\):\s+(\d+\.\d+\s*\(\s*\d+\.\d+\))', content)

    # Create a dictionary to hold the data
    data = {
        'Title': [title_match.group(0) if title_match else None],
        'ATOM PARAMETERS': [atom_params_match.group(1) if atom_params_match else None],
        'Cell parameters': [cell_params_match.group(1).strip() if cell_params_match else None],
        'Overall tem. factor': [f"{overall_tem_factor_match.group(1)} {overall_tem_factor_match.group(2)}" if overall_tem_factor_match else None],
        'R factors': [f"{r_factors_match.group(1)} {r_factors_match.group(2)} {r_factors_match.group(4)}" if r_factors_match else None],
        'Bragg R-factor': [f"{bragg_r_factor_match.group(1)} {bragg_r_factor_match.group(2)} {bragg_r_factor_match.group(3)}" if bragg_r_factor_match else None]
    }

    # Create the DataFrame
    df = pd.DataFrame(data)

    # Organize the 'Title' column
    def organize_title(title):
        if title:
            parts = title.split('-')
            if len(parts) == 2:
                running_number = parts[0].strip()
                temperature = parts[1].strip().replace('K', '').replace(' ', '-')
                if temperature.endswith('-'):
                    temperature = temperature[:-1]
                return running_number, temperature
        return None, None

    # Apply the function and create new columns
    df[['Running Number of Data File', 'Temperature of measurement']] = df['Title'].apply(
        lambda x: pd.Series(organize_title(x))
    )

    # Drop the original 'Title' column
    df.drop(columns=['Title'], inplace=True)

    # Function to parse ATOM PARAMETERS and create a DataFrame
    def parse_atom_parameters(atom_params):
        lines = atom_params.split('\n')
        columns = ['Name', 'x', 'sx', 'y', 'sy', 'z', 'sz', 'B', 'sB', 'occ', 'socc', 'Mult']
        data = []
        for line in lines[1:]:  # Skip the header line
            parts = re.split(r'\s+', line.strip())
            if len(parts) == 12:
                data.append(parts)
        
        atom_df = pd.DataFrame(data, columns=columns)
        
        organized_data = {}
        
        for _, row in atom_df.iterrows():
            name = row['Name']
            organized_data[f'x({name})'] = row['x']
            organized_data[f'ESD(x({name}))'] = row['sx']
            organized_data[f'y({name})'] = row['y']
            organized_data[f'ESD(y({name}))'] = row['sy']
            organized_data[f'z({name})'] = row['z']
            organized_data[f'ESD(z({name}))'] = row['sz']
            organized_data[f'B({name})'] = row['B']
            organized_data[f'ESD(B({name}))'] = row['sB']
            organized_data[f'Occupation factor({name})'] = row['occ']
            organized_data[f'ESD(occupation factor({name}))'] = row['socc']
        
        return pd.DataFrame([organized_data])

    # Apply the parsing function to the 'ATOM PARAMETERS' column and merge with the main DataFrame
    atom_parameters_df = df['ATOM PARAMETERS'].apply(parse_atom_parameters)
    atom_parameters_df = pd.concat(atom_parameters_df.values).reset_index(drop=True)

    # Drop the original 'ATOM PARAMETERS' column and join the new organized DataFrame
    df.drop(columns=['ATOM PARAMETERS'], inplace=True)
    df = pd.concat([df, atom_parameters_df], axis=1)

    # Function to parse and organize 'Cell parameters'
    def organize_cell_parameters(cell_params):
        lines = cell_params.strip().split('\n')
        organized_data = {}
        
        if len(lines) == 6:
            organized_data['Unit cell parameter a'] = lines[0].strip()
            organized_data['Unit cell parameter b'] = lines[1].strip()
            organized_data['Unit cell parameter c'] = lines[2].strip()
            organized_data['Unit cell parameter alpha'] = lines[3].strip()
            organized_data['Unit cell parameter beta'] = lines[4].strip()
            organized_data['Unit cell parameter gamma'] = lines[5].strip()
        
        return pd.DataFrame([organized_data])

    # Apply the function to the 'Cell parameters' column and merge with the main DataFrame
    cell_parameters_df = df['Cell parameters'].apply(organize_cell_parameters)
    cell_parameters_df = pd.concat(cell_parameters_df.values).reset_index(drop=True)

    # Drop the original 'Cell parameters' column and join the new organized DataFrame
    df.drop(columns=['Cell parameters'], inplace=True)
    df = pd.concat([df, cell_parameters_df], axis=1)

    # Function to parse and organize 'Overall tem. factor'
    def organize_overall_tem_factor(overall_tem_factor):
        parts = overall_tem_factor.split()
        if len(parts) == 2:
            return pd.Series({'Bover': parts[0], 'ESD(Bover)': parts[1]})
        return pd.Series({'Bover': None, 'ESD(Bover)': None})

    # Apply the function to the 'Overall tem. factor' column and merge with the main DataFrame
    overall_tem_factor_df = df['Overall tem. factor'].apply(organize_overall_tem_factor)
    df = pd.concat([df, overall_tem_factor_df], axis=1)
    df.drop(columns=['Overall tem. factor'], inplace=True)

    # Function to parse and organize 'R factors'
    def organize_r_factors(r_factors):
        parts = r_factors.split()
        if len(parts) == 4:
            return pd.Series({'Rp': parts[0], 'Rwp': parts[1], 'Chi^2': parts[3]})
        return pd.Series({'Rp': None, 'Rwp': None, 'Chi^2': None})

    # Apply the function to the 'R factors' column and merge with the main DataFrame
    r_factors_df = df['R factors'].apply(organize_r_factors)
    df = pd.concat([df, r_factors_df], axis=1)
    df.drop(columns=['R factors'], inplace=True)

    # Function to parse and organize 'Bragg R-factor'
    def organize_bragg_r_factor(bragg_r_factor):
        parts = bragg_r_factor.split()
        if len(parts) == 3:
            vol_parts = parts[1].split('(')
            fract_parts = parts[2].split('(')
            if len(vol_parts) == 2 and len(fract_parts) == 2:
                return pd.Series({
                    'Bragg R-factor': parts[0],
                    'Unit cell volume': vol_parts[0],
                    'ESD(Unit cell volume)': vol_parts[1].replace(')', ''),
                    'Mass density (calculated)': fract_parts[0],
                    'ESD(Mass density)': fract_parts[1].replace(')', '')
                })
        return pd.Series({
            'Bragg R-factor': None,
            'Unit cell volume': None,
            'ESD(Unit cell volume)': None,
            'Mass density (calculated)': None,
            'ESD(Mass density)': None
        })

    # Apply the function to the 'Bragg R-factor' column and merge with the main DataFrame
    bragg_r_factor_df = df['Bragg R-factor'].apply(organize_bragg_r_factor)
    df = pd.concat([df, bragg_r_factor_df], axis=1)
    df.drop(columns=['Bragg R-factor'], inplace=True)

    return df

# List of .sum files to process
sum_files = [
    r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240751-754.sum',
    r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240754-757.sum',
    r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240757-760.sum',
    r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240760-763.sum',
    r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240763-766.sum'
]

for file_path in sum_files:
    print(f"Processing file: {file_path}")
    df = process_sum_file(file_path)
    if df is not None:
        csv_path = file_path.replace('.sum', '.csv')
        df.to_csv(csv_path, index=False)
        print(f"Processed and saved: {csv_path}")
    else:
        print(f"Skipping file due to error: {file_path}")


Processing file: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240751-754.sum
File not found: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240751-754.sum
Skipping file due to error: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240751-754.sum
Processing file: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240754-757.sum
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240754-757.csv
Processing file: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240757-760.sum
File not found: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240757-760.sum
Skipping file due to error: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240757-760.sum
Processing file: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240760-763.sum
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\240760-763.csv
Processing file: C:\Users\Master.WR\OneDri

In [6]:
import os
import re
import pandas as pd

def extract_sections(content):
    data = {}

    # Extract Title
    title_match = re.search(r"Title:\s*(.+)", content)
    data['Title'] = title_match.group(1).strip() if title_match else None

    # Extract ATOM PARAMETERS section
    atom_params_match = re.search(r"==> ATOM PARAMETERS:\s*\n((?:.*\n)+?)(?=\n\s*==>)", content)
    data['ATOM PARAMETERS'] = atom_params_match.group(1).strip() if atom_params_match else None

    # Extract Cell parameters section
    cell_params_match = re.search(r"Cell parameters\s*:\s*\n((?:\s*\d+\.\d+\s*\d+\.\d+\s*\n)+)", content)
    data['Cell parameters'] = cell_params_match.group(1).strip() if cell_params_match else None
    
    # Extract Overall tem. factor
    overall_tem_factor_match = re.search(r"=> Overall tem\. factor\s*:\s*(\d+\.\d+)\s+(\d+\.\d+)", content)
    data['Overall tem. factor'] = overall_tem_factor_match.group(0).strip() if overall_tem_factor_match else None

    # Extract Rp, Rwp, Rexp, Chi2
    r_factors_match = re.search(r"=> Rp:\s*(\d+\.\d+)\s+Rwp:\s*(\d+\.\d+)\s+Rexp:\s*(\d+\.\d+)\s+Chi2:\s*(\d+\.\d+)", content)
    data['R factors'] = r_factors_match.group(0).strip() if r_factors_match else None

    # Extract Bragg R-factor, Vol, Fract(%)
    bragg_r_factor_match = re.search(r"=> Bragg R-factor:\s*(\d+\.\d+)\s+Vol:\s*(\d+\.\d+)\(\s*\d+\.\d+\)\s+Fract\(%\):\s*(\d+\.\d+)\(\s*\d+\.\d+\)", content)
    data['Bragg R-factor'] = bragg_r_factor_match.group(0).strip() if bragg_r_factor_match else None
    
    return data

def organize_title(title):
    if title:
        parts = title.split('-')
        if len(parts) == 2:
            running_number = parts[0].strip()
            temperature = parts[1].strip().replace('K', '').replace(' ', '-')
            if temperature.endswith('-'):
                temperature = temperature[:-1]
            return running_number, temperature
    return None, None

def parse_atom_parameters(atom_params):
    lines = atom_params.split('\n')
    columns = ['Name', 'x', 'sx', 'y', 'sy', 'z', 'sz', 'B', 'sB', 'occ', 'socc', 'Mult']
    data = []
    for line in lines[1:]:  # Skip the header line
        parts = re.split(r'\s+', line.strip())
        if len(parts) == 12:
            data.append(parts)
    
    atom_df = pd.DataFrame(data, columns=columns)
    
    organized_data = {}
    
    for _, row in atom_df.iterrows():
        name = row['Name']
        organized_data[f'x({name})'] = row['x']
        organized_data[f'ESD(x({name}))'] = row['sx']
        organized_data[f'y({name})'] = row['y']
        organized_data[f'ESD(y({name}))'] = row['sy']
        organized_data[f'z({name})'] = row['z']
        organized_data[f'ESD(z({name}))'] = row['sz']
        organized_data[f'B({name})'] = row['B']
        organized_data[f'ESD(B({name}))'] = row['sB']
        organized_data[f'Occupation factor({name})'] = row['occ']
        organized_data[f'ESD(occupation factor({name}))'] = row['socc']
    
    return pd.DataFrame([organized_data])

def organize_cell_parameters(cell_params):
    lines = cell_params.strip().split('\n')
    organized_data = {}
    
    if len(lines) == 6:
        organized_data['Unit cell parameter a'] = lines[0].strip()
        organized_data['Unit cell parameter b'] = lines[1].strip()
        organized_data['Unit cell parameter c'] = lines[2].strip()
        organized_data['Unit cell parameter alpha'] = lines[3].strip()
        organized_data['Unit cell parameter beta'] = lines[4].strip()
        organized_data['Unit cell parameter gamma'] = lines[5].strip()
    
    return pd.DataFrame([organized_data])

def organize_overall_tem_factor(overall_tem_factor):
    parts = overall_tem_factor.replace('=> Overall tem. factor  :', '').strip().split()
    if len(parts) == 2:
        return pd.Series({'Bover': parts[0], 'ESD(Bover)': parts[1]})
    return pd.Series({'Bover': None, 'ESD(Bover)': None})

def organize_r_factors(r_factors):
    parts = r_factors.replace('=> Rp:', '').replace('Rwp:', '').replace('Rexp:', '').replace('Chi2:', '').split()
    if len(parts) == 4:
        return pd.Series({'Rp': parts[0], 'Rwp': parts[1], 'Chi^2': parts[3]})
    return pd.Series({'Rp': None, 'Rwp': None, 'Chi^2': None})

def organize_bragg_r_factor(bragg_r_factor):
    match = re.search(
        r'=> Bragg R-factor:\s+([\d.]+)\s+Vol:\s+([\d.]+)\s*\(\s*([\d.]+)\s*\)\s+Fract\(%\):\s+([\d.]+)\s*\(\s*([\d.]+)\s*\)',
        bragg_r_factor
    )
    if match:
        RBragg, V, ESD_V, w, ESD_w = match.groups()
        return pd.Series({
            'RBragg': RBragg,
            'Unit cell volume V': V,
            'ESD(V)': ESD_V,
            'Phase fraction w': w,
            'ESD(w)': ESD_w
        })
    return pd.Series({
        'RBragg': None,
        'Unit cell volume V': None,
        'ESD(V)': None,
        'Phase fraction w': None,
        'ESD(w)': None
    })

def process_sum_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            content = file.read()
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None

    extracted_data = extract_sections(content)

    df = pd.DataFrame([extracted_data])

    df[['Running Number of Data File', 'Temperature of measurement']] = df['Title'].apply(
        lambda x: pd.Series(organize_title(x))
    )
    df.drop(columns=['Title'], inplace=True)

    atom_parameters_df = df['ATOM PARAMETERS'].apply(parse_atom_parameters)
    atom_parameters_df = pd.concat(atom_parameters_df.values).reset_index(drop=True)
    df.drop(columns=['ATOM PARAMETERS'], inplace=True)
    df = pd.concat([df, atom_parameters_df], axis=1)

    cell_parameters_df = df['Cell parameters'].apply(organize_cell_parameters)
    cell_parameters_df = pd.concat(cell_parameters_df.values).reset_index(drop=True)
    df.drop(columns=['Cell parameters'], inplace=True)
    df = pd.concat([df, cell_parameters_df], axis=1)

    overall_tem_factor_df = df['Overall tem. factor'].apply(organize_overall_tem_factor)
    df = pd.concat([df, overall_tem_factor_df], axis=1)
    df.drop(columns=['Overall tem. factor'], inplace=True)

    r_factors_df = df['R factors'].apply(organize_r_factors)
    df = pd.concat([df, r_factors_df], axis=1)
    df.drop(columns=['R factors'], inplace=True)

    bragg_r_factor_df = df['Bragg R-factor'].apply(organize_bragg_r_factor)
    df = pd.concat([df, bragg_r_factor_df], axis=1)
    df.drop(columns=['Bragg R-factor'], inplace=True)

    return df

# Directory containing .sum files
directory = r'C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file'

# List .sum files in the directory
sum_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.sum')]

# Process each file and save to CSV
for file_path in sum_files:
    df = process_sum_file(file_path)
    if df is not None:
        output_file = file_path.replace('.sum', '.csv')
        df.to_csv(output_file, index=False)
        print(f"Processed and saved: {output_file}")
    else:
        print(f"Skipping file due to error: {file_path}")


Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240748-751.csv
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240751-754 .csv
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240754-757.csv
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240757-760 .csv
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240760-763.csv
Processed and saved: C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\sum_file\240763-766.csv
